# Secure Federated Learning Demo (Google Colab)
## Secure Federated Linear Regression (SFLR) and Secure Federated Tree Boosting (SFTB)

This notebook demonstrates simplified simulations of:
- Secure Federated Linear Regression (SFLR)
- Conceptual Secure Federated Tree Boosting (SFTB)

The goal is to illustrate **Vertical Federated Learning (VFL)** where different parties own different features of the same users without sharing raw data.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor


## 2. Simulated Vertical Federated Dataset

Two parties hold different features of the same samples.

In [ ]:
np.random.seed(42)

n_samples = 100

# Party A features
XA = np.random.rand(n_samples,2)

# Party B features
XB = np.random.rand(n_samples,2)

# True weights
wA_true = np.array([2,3])
wB_true = np.array([4,5])

# Generate labels
y = XA.dot(wA_true) + XB.dot(wB_true) + np.random.normal(0,0.1,n_samples)

print("Party A feature shape:", XA.shape)
print("Party B feature shape:", XB.shape)

Party A feature shape: (100, 2)
Party B feature shape: (100, 2)


## 3. Secure Federated Linear Regression (SFLR)

Each party computes partial predictions locally and shares **intermediate values only**.

In [ ]:
# Initialize weights
WA = np.random.rand(2)
WB = np.random.rand(2)

lr = 0.05
epochs = 200

for epoch in range(epochs):

    # Local computations
    zA = XA.dot(WA)
    zB = XB.dot(WB)

    # Secure aggregation (simulated)
    pred = zA + zB

    error = pred - y

    # Local gradients
    gradA = XA.T.dot(error) / n_samples
    gradB = XB.T.dot(error) / n_samples

    WA -= lr * gradA
    WB -= lr * gradB

pred_final = XA.dot(WA) + XB.dot(WB)

mse = mean_squared_error(y, pred_final)

print("Final MSE:", mse)
print("Learned weights Party A:", WA)
print("Learned weights Party B:", WB)

Final MSE: 0.05347102306928687
Learned weights Party A: [2.42288284 3.27631248]
Learned weights Party B: [3.94581192 4.39850133]


## 4. Simplified Secure Federated Tree Boosting Concept

We simulate gradient boosting where gradients are computed by the **active party** (who has labels) and trees are trained using combined encrypted statistics.

This is a simplified conceptual version of **SecureBoost**.

In [ ]:
# Combine features logically (without sharing raw data in real systems)
X_combined = np.hstack([XA, XB])

n_estimators = 10
learning_rate = 0.3

pred = np.zeros(n_samples)
trees = []

for i in range(n_estimators):

    # Active party computes gradients
    residual = y - pred

    # Train tree on residuals
    tree = DecisionTreeRegressor(max_depth=3)
    tree.fit(X_combined, residual)

    update = tree.predict(X_combined)

    pred += learning_rate * update

    trees.append(tree)

mse_boost = mean_squared_error(y, pred)

print("Boosted Model MSE:", mse_boost)

Boosted Model MSE: 0.12490217391838986


## 5. Key Takeaways

- Vertical Federated Learning allows organizations to collaborate without sharing raw data.
- SFLR trains regression models using distributed features.
- SFTB enables privacy‑preserving decision tree boosting.
- Real implementations use frameworks like **FATE, Flower, and SecureBoost**.